<a href="https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Copy the real table path pattern from notebook 03 and from the dataset page's file list
TABLE = "hf://datasets/FlyRank/internship-warehouse/___/*.parquet"   # fill in

print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{TABLE}', hive_partitioning=true)"))

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 1) The contract

1. **One row means:** one ___ (e.g. one page on one day).
2. **Table(s) used:** ___
3. **Time window:** month=2026-03 for development; June 2026 is kept sealed as the test month.
4. **What I would predict or rank (label or proxy):** ___
5. **One thing I deliberately exclude:** ___ (because ___).

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT ___key_cols___, COUNT(*) AS n
FROM read_parquet('{TABLE}', hive_partitioning=true)
WHERE month = '2026-03'
GROUP BY ___key_cols___
HAVING COUNT(*) > 1
LIMIT 10
""").df()

In [ ]:
con.sql(f"""
SELECT COUNT(*) AS rows, MIN(___date_col___) AS first_day, MAX(___date_col___) AS last_day
FROM read_parquet('{TABLE}', hive_partitioning=true)
WHERE month = '2026-03'
""").df()

In [ ]:
con.sql(f"""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE ___availability_flag___ IS TRUE) AS available_rows
FROM read_parquet('{TABLE}', hive_partitioning=true)
WHERE month = '2026-03'
""").df()

In [ ]:
df = con.sql(f"""
SELECT ___id___, ___feat1___, ___feat2___, ___feat3___, ___feat4___, ___feat5___, ___label___
FROM read_parquet('{TABLE}', hive_partitioning=true)
WHERE month = '2026-03' AND ___availability_flag___ IS TRUE
LIMIT 50000
""").df()
df.head()

In [ ]:
### Features and when each is knowable

- **___feat1___:** knowable at the decision moment because ___.
- **___feat2___:** knowable at the decision moment because ___.
- **___feat3___:** knowable at the decision moment because ___.
- **___feat4___:** knowable at the decision moment because ___.
- **___feat5___:** knowable at the decision moment because ___.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

FEATS = ["___feat1___", "___feat2___", "___feat3___", "___feat4___", "___feat5___"]
LABEL = "___label___"
d = df.dropna(subset=FEATS + [LABEL])

Xtr, Xte, ytr, yte = train_test_split(d[FEATS], d[LABEL], test_size=0.3, random_state=0)
m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xtr, ytr)
honest = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
print("Honest score (AUC):", round(honest, 3))

In [ ]:
# A column derived from the label itself (a deliberate mistake)
d = d.copy()
d["leak"] = d[LABEL] * 1.0

Xtr, Xte, ytr, yte = train_test_split(d[FEATS + ["leak"]], d[LABEL], test_size=0.3, random_state=0)
m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xtr, ytr)
leaky = roc_auc_score(yte, m.predict_proba(Xte)[:, 1])
print("Leaky score (AUC):", round(leaky, 3))

In [ ]:
d = d.drop(columns=["leak"])
print("Back to honest score:", round(honest, 3), "| leaky score was:", round(leaky, 3))

## The trap

Adding a column made from the label pushed the score toward perfect (___ vs ___ honestly). That jump is a warning, not a win: the model was handed the answer. I removed the column and kept the honest score of ___.

## 4) One named limitation of my slice

___ (for example: only one month, only available rows, anonymized so some columns are missing, or the label is a proxy.)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.